In [1]:
import kagglehub

#download dataset

path= kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")

print("Dataset path:", path)

Using Colab cache for faster access to the 'new-plant-diseases-dataset' dataset.
Dataset path: /kaggle/input/new-plant-diseases-dataset


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [3]:
#list main folder

print(os.listdir(path))

['New Plant Diseases Dataset(Augmented)', 'new plant diseases dataset(augmented)', 'test']


In [4]:

# Assuming 'path' is the folder returned from kagglehub
train = os.path.join(path, 'New Plant Diseases Dataset(Augmented)', 'New Plant Diseases Dataset(Augmented)', 'train')
validation = os.path.join(path, 'New Plant Diseases Dataset(Augmented)', 'New Plant Diseases Dataset(Augmented)', 'valid')



In [5]:
test = os.path.join(path,'test', 'test')

In [6]:
test

'/kaggle/input/new-plant-diseases-dataset/test/test'

In [7]:
print("Train exists:", os.path.exists(train))
print("Validation exists:", os.path.exists(validation))
print("Test exists:", os.path.exists(test))


Train exists: True
Validation exists: True
Test exists: True


**Define transform**

Torchvision help to resize, normalize augument data

In [8]:
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch

In [9]:
# Load with only resizing and tensor conversion

train_transforms = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor()
     ])

train_dataset = ImageFolder(train, transform= train_transforms)

In [10]:
loader = DataLoader(train_dataset,batch_size =32, shuffle= True)

In [11]:
#Normalize

mean = 0.0
std = 0.0
total_images = 0
for images, _ in loader:
    batch_samples = images.size(0)  # get batch size i.e 32
    images = images.view(batch_samples, images.size(1), -1) # From shape [32, 3, 224, 224] → [32, 3, 50176]
    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    total_images += batch_samples

mean = mean / total_images
std = std / total_images
print(mean)
print(std)



tensor([0.4760, 0.5004, 0.4266])
tensor([0.1775, 0.1509, 0.1960])


In [12]:
from torchvision import transforms

# ImageNet-style mean/std (or use your own)
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Training transform (with augmentation)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Validation/test transform (no augmentation)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])


In [13]:
# Make sure `train_dir` is set correctly
train_dataset = ImageFolder(train, transform=train_transforms)
val_dataset = ImageFolder(validation, transform=val_transforms)

*DATA loader*

In [14]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [15]:
train_loader

**CNN** **Model**

In [16]:
import torch.nn as nn
import torch.nn.functional as F

class PlantDiseaseCNN(nn.Module):
    def __init__(self, num_classes):
        super(PlantDiseaseCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)

        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(128 * 28 * 28, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)  # Flatten
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x


In [17]:
num_classes = len(train_dataset.classes)
model = PlantDiseaseCNN(num_classes=num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [18]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [20]:
import torch

num_epochs = 10  # Start small, increase later



for epoch in range(num_epochs):
    model.train()  # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Stats
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Epoch results
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f} Accuracy: {epoch_acc:.2f}%")


Epoch [1/10] Loss: 1.6660 Accuracy: 49.80%
Epoch [2/10] Loss: 0.8135 Accuracy: 74.64%
Epoch [3/10] Loss: 0.5850 Accuracy: 81.66%
Epoch [4/10] Loss: 0.4853 Accuracy: 84.71%
Epoch [5/10] Loss: 0.4258 Accuracy: 86.37%
Epoch [6/10] Loss: 0.3792 Accuracy: 88.05%
Epoch [7/10] Loss: 0.3390 Accuracy: 89.30%
Epoch [8/10] Loss: 0.3186 Accuracy: 89.91%
Epoch [9/10] Loss: 0.3069 Accuracy: 90.38%
Epoch [10/10] Loss: 0.2818 Accuracy: 91.10%


In [21]:
model.eval()


PlantDiseaseCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=100352, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=38, bias=True)
)

In [22]:
example_input = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    traced_model = torch.jit.trace(model, example_input)


In [23]:
original_output = model(example_input)
traced_output = traced_model(example_input)

# Compare difference (optional)
print("Difference:", torch.abs(original_output - traced_output).max())


Difference: tensor(0., device='cuda:0', grad_fn=<MaxBackward1>)


In [24]:
scripted_model = torch.jit.script(model)
scripted_model.save("plant_disease_model_mobile.pt")


In [25]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the saved scripted model
model = torch.jit.load("plant_disease_model_mobile.pt")
model = model.to(device)
model.eval()  # Set model to evaluation mode


RecursiveScriptModule(
  original_name=PlantDiseaseCNN
  (conv1): RecursiveScriptModule(original_name=Conv2d)
  (pool1): RecursiveScriptModule(original_name=MaxPool2d)
  (conv2): RecursiveScriptModule(original_name=Conv2d)
  (pool2): RecursiveScriptModule(original_name=MaxPool2d)
  (conv3): RecursiveScriptModule(original_name=Conv2d)
  (pool3): RecursiveScriptModule(original_name=MaxPool2d)
  (dropout): RecursiveScriptModule(original_name=Dropout)
  (fc1): RecursiveScriptModule(original_name=Linear)
  (fc2): RecursiveScriptModule(original_name=Linear)
)

In [26]:
from torchvision import transforms

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),              # Resize to match training input
    transforms.ToTensor(),                      # Convert to tensor
    transforms.Normalize(                       # Normalize like ImageNet/pretrained models
        mean=[0.485, 0.456, 0.406],             # Red, Green, Blue means
        std=[0.229, 0.224, 0.225]
    )
])


In [27]:
test_dataset = ImageFolder(validation, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [28]:
model.eval()  # Very important

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total
print(f"✅ Test Accuracy on validation set: {test_acc:.2f}%")


✅ Test Accuracy on validation set: 95.16%


In [29]:
from PIL import Image
import torch
import torchvision.transforms as transforms

# Same transforms as used during training/testing
predict_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [30]:
# Function to predict class
def predict_image(image_path, model, class_names):
    model.eval()

    # Load and preprocess image
    image = Image.open(image_path).convert("RGB")
    image = predict_transforms(image).unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        outputs = model(image)
        _, predicted = torch.max(outputs, 1)

    predicted_class = class_names[predicted.item()]
    return predicted_class

In [32]:
img_path = "/content/test_image.jpg"  # Replace with image path from mobile app
prediction = predict_image(img_path, model, train_dataset.classes)

print("🌿 Predicted Disease:", prediction)


🌿 Predicted Disease: Apple___Apple_scab
